In [ ]:
import pandas as pd
import numpy as np
from google.colab import files
df = pd.read_csv('/NIJ_s_Recidivism_Challenge_Full_Dataset.csv')

# Looking at the first 10 rows of the dataset
print(df.head(10))
#Checking for null vlaues
print(df.isna().any())

   ID Gender   Race Age_at_Release  Residence_PUMA Gang_Affiliated  \
0   1      M  BLACK          43-47              16           False   
1   2      M  BLACK          33-37              16           False   
2   3      M  BLACK    48 or older              24           False   
3   4      M  WHITE          38-42              16           False   
4   5      M  WHITE          33-37              16           False   
5   6      M  WHITE          38-42              17           False   
6   7      M  BLACK    48 or older              18           False   
7   8      M  BLACK          38-42              16           False   
8   9      F  BLACK          43-47               5             NaN   
9  10      M  BLACK          43-47              16           False   

   Supervision_Risk_Score_First Supervision_Level_First  \
0                           3.0                Standard   
1                           6.0             Specialized   
2                           7.0                    H

Average Recidivism Rate (within 3 years) Based on Program Attendance

In [ ]:
prog_df = (df.groupby('Program_Attendances')['Recidivism_Within_3years'].mean() * 100).round(1).reset_index(name='recidivism_rate')
print(prog_df)

   Program_Attendances  recidivism_rate
0                    0             59.1
1                    1             67.7
2           10 or more             47.1
3                    2             65.7
4                    3             66.1
5                    4             59.9
6                    5             60.7
7                    6             52.5
8                    7             57.0
9                    8             60.9
10                   9             59.1


Impact of Employment on Avg Recidivism Rate

In [ ]:
#Checking if nulls for 'Percent_Days_Employed' correlate with employment exemption
print(pd.crosstab(df['Employment_Exempt'], df['Percent_Days_Employed'].isnull()))

cleaned_df = df.dropna(subset=['Percent_Days_Employed']).copy()

# Defining the bins and labels.
# Since Percent_Days_Employed goes from 0.0 to 1.0 (representing 0% to 100%):
bins = [-0.01, 0.0, 0.25, 0.50, 0.75, 1.0]
labels = ['0%', '1–25%', '26–50%', '51–75%', '76–100%']

# Create the binned column
cleaned_df['Employment_Group'] = pd.cut(cleaned_df['Percent_Days_Employed'], bins=bins, labels=labels)

# Group by the new bins and calculate the average recidivism rate
emp_df = (cleaned_df.groupby('Employment_Group', observed=False)['Recidivism_Within_3years'].mean() * 100).round(1).reset_index(name='recidivism_rate')
print(emp_df)

Percent_Days_Employed  False  True 
Employment_Exempt                  
False                  22015    384
True                    3358     78
  Employment_Group  recidivism_rate
0               0%             67.2
1            1–25%             79.8
2           26–50%             73.6
3           51–75%             57.0
4          76–100%             43.6


Impact of Substance Use on Avg Recidivism Rate

In [ ]:
# Boolean evaluates to true if at least one test result is > 0
df['Postive_Substance_Test'] = (df['DrugTests_THC_Positive'] > 0) | (df['DrugTests_Cocaine_Positive'] > 0) | (df['DrugTests_Meth_Positive'] > 0) | (df['DrugTests_Other_Positive'] > 0)
# Boolean evaluates to true if the selected result is null (if tests are missing it should be for all substances)
df['Missing_Subtance_Tests'] = df['DrugTests_THC_Positive'].isna()


In [ ]:
# Defining logical conditions
conditions = [
    df['Missing_Subtance_Tests'],     # Condition 1: Missing / Un-tested
    df['Postive_Substance_Test'],     # Condition 2: Tested & At least one positive
]

# Defining the corresponding labels
choices = [
    'Missing / Un-tested',
    'Tested - Positive',
]

# 'default' handles everyone who didn't trigger either above
# (i.e., not missing AND not positive -> Tested - All Clean)
df['Drug_Test_Status'] = np.select(conditions, choices, default='Tested - Clean')

drug_summary = df.groupby('Drug_Test_Status')['Recidivism_Within_3years'].agg(
    recidivism_rate='mean',
    headcount='count'
).reset_index()

# Format the rate as a clean percentage for presentation
drug_summary['recidivism_rate'] = (drug_summary['recidivism_rate'] * 100).round(1)

print(drug_summary)


      Drug_Test_Status  recidivism_rate  headcount
0  Missing / Un-tested             67.6       5172
1       Tested - Clean             50.2      12024
2    Tested - Positive             62.2       8639


Impact of Facility Type on Recidivism Rates

In [ ]:
cy_df = pd.read_csv('Calendar Year (CY) Reconviction Rates (2011–2022 Releases) - Recidivism case study.csv')

# Defining the key facility categories to evaluate
target_facilities = [
    'Transition centers',
    'County CI',
    'All inmate facilities **',
    'State prison, IBCs',
    'Private prisons'
]

# Filtering the dataframe to just those facility types
facility_comparison = cy_df[cy_df['Facility Type'].isin(target_facilities)].copy()

# Identifying the year columns (e.g., '2011' through '2022')
year_cols = [col for col in cy_df.columns if col.isdigit()]

# Ensuring year columns are numeric (converting any potential strings/whitespace)
facility_comparison[year_cols] = facility_comparison[year_cols].apply(pd.to_numeric, errors='coerce')

# Calculating multi-year metrics: Historical Mean, Min, and Max
facility_comparison['Multi_Year_Mean'] = facility_comparison[year_cols].mean(axis=1).round(1)
facility_comparison['10_Year_Min'] = facility_comparison[year_cols].min(axis=1).round(1)
facility_comparison['10_Year_Max'] = facility_comparison[year_cols].max(axis=1).round(1)

# 6. Display a clean summary of latest year vs. historical average
summary_table = facility_comparison[[
    'Facility Type',
    '2022',
    'Multi_Year_Mean',
    '10_Year_Min',
    '10_Year_Max'
]].sort_values(by='2022')

print(summary_table.to_string(index=False))


           Facility Type  2022  Multi_Year_Mean  10_Year_Min  10_Year_Max
      Transition centers  12.3             17.2         12.2         20.7
All inmate facilities **  31.1             27.1         23.8         31.1
         Private prisons  32.3             29.3         25.3         32.5
               County CI  32.8             27.7         22.4         32.8
      State prison, IBCs  33.8             28.1         25.1         33.8


Export Summaries to Multi-tab Excel Workbook

In [ ]:
excel_filename = 'Recidivism_Case_Study_Summary.xlsx'

# Write your clean DataFrames straight to their respective sheet tabs
with pd.ExcelWriter(excel_filename) as writer:
    prog_df.to_excel(writer, sheet_name='Program_Attendance', index=False)
    emp_df.to_excel(writer, sheet_name='Employment_Tiers', index=False)
    drug_summary.to_excel(writer, sheet_name='Substance_Use', index=False)
    summary_table.to_excel(writer, sheet_name='Facility_Comparison', index=False)

print(f"Excel workbook '{excel_filename}' successfully compiled and downloaded!")

# Automatically trigger the local download in your browser
files.download(excel_filename)


Excel workbook 'Recidivism_Case_Study_Summary.xlsx' successfully compiled and downloaded!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>